In [1]:
import numpy as np
import torch
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [2]:
!pwd

/isaac-sim/memoryPerceiver/memory_perceiver/main


In [3]:
from os.path import join, dirname
import os
import sys

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 1

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [6]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_parts)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            32,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = batch_size
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_parts.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [7]:
model, queries = create_model()

inputs={}
inputs['video']=torch.randn(batch_size, 3, nb_frames, 256, 456)
inputs['audio']=torch.randn(batch_size, nb_frames*1920)
inputs['observations']=torch.randn(batch_size, nb_frames, 4)

batch = input_to_tensor(inputs, model.modality, cf.body_parts)

In [8]:
with torch.no_grad():
    x, aux_loss = model(batch.to(device), queries = queries)

In [12]:
numEnvs=8

In [14]:
"fdsf"+str(numEnvs)

TypeError: can only concatenate str (not "int") to str

In [17]:
stack = []
for i in range(32):
    stack.append(torch.randn(8,4))
(rearrange(stack, 't b d -> b t d')).size()

torch.Size([8, 32, 4])

In [9]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    cfg_dict = omegaconf_to_dict(cfg)
    print_dict(cfg_dict)

    headless = cfg.headless
    render = not headless

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    save_step = 160
    numEnvs = 8
    obs_buffer = []
    reward_buffer = []
    action_buffer = []
    while env._simulation_app.is_running():
        if env._world.is_playing():
            if env._world.current_time_step_index == 0:
                env._world.reset(soft=True)

            # Observations: 'cartpole_view' 'obs_buf' numEnvs, 4

            obs = env._task.get_observations()['cartpole_view']['obs_buf']
            obs_buffer.append(obs)
            if(len(obs_buffer)>32):
                 obs_buffer.pop(0)

            # Reward numEnvs, 1

            reward = env._task.calculate_metrics()
            reward_buffer.append(reward)
            if(len(reward_buffer)>32):
                 reward_buffer.pop(0)

            # Process input

            inputs={}
            inputs['video']=torch.randn(numEnvs, 3, nb_frames, 256, 456)
            inputs['audio']=torch.randn(numEnvs, nb_frames*1920)
            inputs['observations']=rearrange(obs_buffer, 't b d -> b t d')
            batch = input_to_tensor(inputs, model.modality, cf.body_parts)

            # 

            # media['video'] = default_video
            # media['audio'] = default_audio
            #
            # if env.sim_frame_count % save_step == 0 and len(obs_buffer) >= save_step:
            # Save mem body, audio, video


            # if(len(action_buffer) == 0 and len(obs_buffer) >= 8):
                # obs.append(get camera view())
                # obs.append(audio)
                # with torch.no_grad():
                    # action, voice = memory_perceiver(obs_buffer[-8:])
                # action_buffer.append(action)
            # if(len(action_buffer)>0):
                # action = action_buffer.pop(0)
            # else:
                # action = torch.randn(512, 1)

            # numEnvs, 1
            action = torch.tensor(np.array([env.action_space.sample() for _ in range(env.num_envs)]), device=task.rl_device)
            env._task.pre_physics_step(action)
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()
        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


ConfigCompositionException: Could not override 'numEnvs'.
To append to your config use +numEnvs=512